# Traffic Data Dictionary

| Column Name | Description |
| :--- | :--- |
| **Index** | Represents the unique identification of a datapoint. |
| **geohash** | Represents geographic information regarding a specific place. |
| **day** | Represents the day when the information was recorded. |
| **timestamp** | Represents the timestamp of the record inserted into the system. |
| **RoadType** | Represents the type of road in the nearby location. |
| **NumberofLanes** | Represents the number of roads/lanes present in the location. |
| **LargeVehicles** | Represents whether large vehicles are permitted on the specific roads/lanes. |
| **Landmarks** | Represents whether there are any landmarks near the location. |
| **Temperature** | Represents the temperature of the place. |
| **Weather** | Represents the weather conditions of the place. |
| **demand** | Represents the traffic demand at the specific timestamp. |

In [25]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score, root_mean_squared_error

train_file_path = '/Users/pashantraj/Desktop/flipkart/dataset/train.csv'

In [26]:
df_full = pd.read_csv(train_file_path)
base_date = pd.to_datetime('2024-01-01')
df_full['datetime'] = (base_date
    + pd.to_timedelta(df_full['day'], unit='D')
    + pd.to_timedelta(df_full['timestamp'] + ':00')
)
df_full = df_full.sort_values(['geohash', 'datetime']).reset_index(drop=True)

In [27]:
def add_lag_features(df):
    grp = df.groupby('geohash')['demand']
    df['lag_1']  = grp.shift(1)    # 15 min ago
    df['lag_2']  = grp.shift(2)    # 30 min ago
    df['lag_4']  = grp.shift(4)    # 1 hour ago
    df['lag_8']  = grp.shift(8)    # 2 hours ago
    df['lag_96'] = grp.shift(96)   # same time yesterday (96 × 15 min)
    df['roll_mean_4']  = grp.transform(lambda x: x.shift(1).rolling(4,  min_periods=1).mean())
    df['roll_mean_16'] = grp.transform(lambda x: x.shift(1).rolling(16, min_periods=1).mean())
    df['roll_std_4']   = grp.transform(lambda x: x.shift(1).rolling(4,  min_periods=1).std().fillna(0))
    lag_cols = ['lag_1','lag_2','lag_4','lag_8','lag_96',
                'roll_mean_4','roll_mean_16','roll_std_4']
    df[lag_cols] = df[lag_cols].fillna(0)   # only NaN at very start of each geohash
    return df

df_full = add_lag_features(df_full)

In [28]:
# ──────────────────────────────────────────────
# STEP 3 — Temporal split AFTER lag computation
# ──────────────────────────────────────────────
cutoff_day = df_full['day'].quantile(0.80)
df_train_split = df_full[df_full['day'] <= cutoff_day].copy()
df_val_split   = df_full[df_full['day'] >  cutoff_day].copy()
master_geohashes = df_full['geohash'].unique().tolist()

print(f"Train rows: {len(df_train_split):,}  |  Val rows: {len(df_val_split):,}")
print(f"Train days: {df_train_split['day'].min()}–{df_train_split['day'].max()} | "
      f"Val days: {df_val_split['day'].min()}–{df_val_split['day'].max()}")

Train rows: 69,427  |  Val rows: 7,872
Train days: 48–48 | Val days: 49–49


In [29]:
# ──────────────────────────────────────────────
# Feature engineering (shared logic)
# Lags already added; just encode + impute
# ──────────────────────────────────────────────
def feature_engineer(df, master_geohashes, global_fallbacks=None,
                     geohash_target_map=None, global_demand_mean=None,
                     is_train=True):
    df = df.copy()

    if is_train:
        global_fallbacks = {
            'RoadType':    df['RoadType'].mode()[0] if not df['RoadType'].mode().empty else 'Main Road',
            'Temperature': df['Temperature'].mean(),
            'Weather':     df['Weather'].mode()[0] if not df['Weather'].mode().empty else 'Sunny',
        }

    # Impute RoadType
    df['RoadType'] = df['RoadType'].fillna(
        df.groupby('geohash')['RoadType'].transform(
            lambda x: x.mode()[0] if not x.mode().empty else np.nan))
    df['RoadType'] = df['RoadType'].fillna(global_fallbacks['RoadType'])

    # Impute Temperature (time-indexed rolling)
    df = df.sort_values(['geohash', 'datetime']).set_index('datetime')
    df['Temperature'] = df.groupby('geohash')['Temperature'].transform(
        lambda x: x.fillna(x.rolling('120min', min_periods=1).mean()))
    df['Temperature'] = df['Temperature'].fillna(global_fallbacks['Temperature'])

    # Impute Weather
    df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda x: x.ffill(limit=4))
    df['Weather'] = df['Weather'].fillna(global_fallbacks['Weather'])
    df = df.reset_index()

    # ── Target encode geohash (mean demand per location) ──
    # Computed on train only; applied to both
    if is_train:
        geohash_target_map = df.groupby('geohash')['demand'].mean()
        global_demand_mean = df['demand'].mean()
    df['geohash_mean_demand'] = (
        df['geohash'].map(geohash_target_map).fillna(global_demand_mean))

    # ── Standard encoding ──
    df['Landmarks']     = df['Landmarks'].map({'No': 0, 'Yes': 1})
    df['LargeVehicles'] = df['LargeVehicles'].map({'Not Allowed': 0, 'Allowed': 1})
    df = pd.get_dummies(df, columns=['RoadType', 'Weather'], drop_first=False)
    for col in df.select_dtypes('bool').columns:
        df[col] = df[col].astype(int)

    # ── Time features ──
    df['time_sin']    = np.sin(2 * np.pi * (df['datetime'].dt.hour * 60 + df['datetime'].dt.minute) / 1440)
    df['time_cos']    = np.cos(2 * np.pi * (df['datetime'].dt.hour * 60 + df['datetime'].dt.minute) / 1440)
    df['hour']        = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

    # Geohash as native XGBoost category
    df['geohash'] = pd.Categorical(df['geohash'], categories=master_geohashes)

    df = df.drop(columns=['datetime', 'day', 'timestamp', 'Index'], errors='ignore')

    return df, global_fallbacks, geohash_target_map, global_demand_mean

In [30]:
# ── Process train ──
ml_train, global_fallbacks, geohash_target_map, global_demand_mean = feature_engineer(
    df_train_split, master_geohashes, is_train=True)

y_train = ml_train['demand']
X_train = ml_train.drop(columns=['demand'])

# ── Process val (reuse train's encoding maps) ──
ml_val, _, _, _ = feature_engineer(
    df_val_split, master_geohashes,
    global_fallbacks=global_fallbacks,
    geohash_target_map=geohash_target_map,
    global_demand_mean=global_demand_mean,
    is_train=False)

y_val = ml_val['demand']
X_val = ml_val.drop(columns=['demand'])

# Align columns: add any missing val columns with 0
for col in X_train.columns:
    if col not in X_val.columns:
        X_val[col] = 0
X_val = X_val[X_train.columns]

# ──────────────────────────────────────────────
# STEP 5 — Train
# ──────────────────────────────────────────────
model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=75,
    enable_categorical=True,
)

model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          verbose=100)

preds = model.predict(X_val)
r2   = r2_score(y_val, preds)
rmse = root_mean_squared_error(y_val, preds)
print(f"\nValidation R²:   {r2:.4f}")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Best iteration:  {model.best_iteration}")

fi = pd.Series(model.feature_importances_, index=X_train.columns).nlargest(10)
print("\nTop 10 features:\n", fi)


[0]	validation_0-rmse:0.14185
[100]	validation_0-rmse:0.03000
[200]	validation_0-rmse:0.02923
[300]	validation_0-rmse:0.02927
[325]	validation_0-rmse:0.02928

Validation R²:   0.9593
Validation RMSE: 0.0292
Best iteration:  250

Top 10 features:
 RoadType_Highway        0.772452
RoadType_Residential    0.114521
lag_1                   0.038370
roll_mean_4             0.024215
lag_2                   0.016877
RoadType_Street         0.013522
LargeVehicles           0.012059
roll_mean_16            0.003416
geohash_mean_demand     0.000843
NumberofLanes           0.000652
dtype: float32
